In [98]:
import pandas as pd
import matplotlib.pyplot as plt
from src.data import *
from src.post_process_functions import *
from src.data_format import format_output, LIST_COLS

In [119]:
cols_to_open = ["uid", "disasterType", "appealCode", "reportDate", "extraction_error","processing_error"]
dropped_df = pd.read_parquet(DATA_IN_JSONS / "no_text_dropped_preproc_text_sel_gaps_merged_df_with_clean_text_all-all_reports_with_pdf_status_020726_with_clean_text_v090726.parquet") 
ymin = 2016
ymax = 2025
dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year
dropped_df = dropped_df.loc[(dropped_df["reportYear"]>=ymin) & (dropped_df["reportYear"]<=ymax)]
dropped_df = dropped_df.rename(columns={"reportDate": "reportDate"})

# final_df = pd.read_parquet(DATA_IN_JSONS / "no_text_preproc_text_sel_gaps_merged_df_with_clean_text_all-all_reports_with_pdf_status_020726_with_clean_text_v090726.parquet")
final_df = pd.read_csv(DATA_IN_JSONS / "preproc_text_sel_gaps_df_with_clean_text_all_combined_v300626_v060726.csv")
final_df = final_df.drop_duplicates(subset=["appealCode", "reportDate"])
final_df["reportYear"] = pd.to_datetime(final_df["reportDate"], errors="coerce", infer_datetime_format=True).dt.year
final_df = final_df.loc[(final_df["reportYear"]>=ymin) & (final_df["reportYear"]<=ymax)]

C:\Users\lhasbini\AppData\Local\Temp\ipykernel_10436\3796781569.py:5: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_10436\3796781569.py:5: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dropped_df["reportYear"] = pd.to_datetime(dropped_df["date"], errors="coerce", infer_datetime_format=True).dt.year
C:\Users\lhasbini\AppData\Local\Temp\ipykernel_10436\3796781569.py:12: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is 

In [120]:
# joined_df = pd.concat([final_df, dropped_df.loc[dropped_df.processing_error!="invalid_appealType"]], ignore_index=True)
joined_df = pd.merge(final_df, dropped_df, how="outer", on=["appealCode", "reportDate"], suffixes=("", "_dropped"))
joined_df = joined_df[["appealCode", "reportDate", "disasterType", "processing_error"]].drop_duplicates()

# Kept df 
kept_df = joined_df.loc[joined_df.processing_error.isna()]

In [121]:
joined_df.processing_error.value_counts()

processing_error
invalid appealType    1529
no natural hazard      471
no text                  3
not in English           3
Name: count, dtype: int64

In [122]:
print("Total number of reports ", len(joined_df), " Number of unique appeals", joined_df["appealCode"].nunique())
print("In final kept db, Total number of reports ", len(kept_df), " Number of unique appeals", kept_df["appealCode"].nunique())

Total number of reports  4139  Number of unique appeals 1972
In final kept db, Total number of reports  2133  Number of unique appeals 778


In [ ]:
# Find the number of unique appealCodes which are dropped because of "no natural hazard" and are not present anymore in the final_df
dropped_no_appealType = joined_df[
    (joined_df["processing_error"] == "invalid appealType") 
]
print(f"Number of unique reports {dropped_no_appealType.shape[0]} dropped because of invalid appealType")
print(f"Number of unique appealCodes {dropped_no_appealType['appealCode'].nunique()} dropped because of invalid appealType")

# Look for the appeals still present in the final_df after dropping the reports with invalid appealType
df_dropped_no_appealType = joined_df.loc[joined_df.processing_error!="invalid appealType"]

Number of unique reports 1529 dropped because of invalid appealType
Number of unique appealCodes 1529 dropped because of invalid appealType


In [137]:
print(f"Number of unique reports {df_dropped_no_appealType.shape[0]} after dropping reports with invalid appealType")
print(f"Number of unique appealCodes {df_dropped_no_appealType['appealCode'].nunique()} after dropping reports with invalid appealType")

Number of unique reports 2610 after dropping reports with invalid appealType
Number of unique appealCodes 1221 after dropping reports with invalid appealType


In [ ]:
# Find the number of unique appealCodes which are dropped because of "no natural hazard" and are not present anymore in the final_df
dropped_no_natural_hazard = df_dropped_no_appealType[
    (df_dropped_no_appealType["processing_error"] == "no natural hazard") 
]
print(f"Number of unique reports {dropped_no_natural_hazard.shape[0]} dropped because of no natural hazard")
print(f"Number of unique appealCodes {dropped_no_natural_hazard['appealCode'].nunique()} dropped because of no natural hazard")

# Look for the appeals still present in the final_df after dropping the reports with invalid appealType
df_dropped_no_natural_hazard = df_dropped_no_appealType.loc[df_dropped_no_appealType.processing_error!="no natural hazard"]

Number of unique reports 471 dropped because of no natural hazard
Number of unique appealCodes 471 dropped because of no natural hazard


In [140]:
print(f"Number of unique reports {df_dropped_no_natural_hazard.shape[0]} after dropping reports with invalid appealType")
print(f"Number of unique appealCodes {df_dropped_no_natural_hazard['appealCode'].nunique()} after dropping reports with invalid appealType")

Number of unique reports 2139 after dropping reports with invalid appealType
Number of unique appealCodes 781 after dropping reports with invalid appealType


In [ ]:
# Find the number of unique appealCodes which are dropped because of "no natural hazard" and are not present anymore in the final_df
dropped_no_text = df_dropped_no_natural_hazard[
    (df_dropped_no_natural_hazard["processing_error"] == "no text") 
]
print(f"Number of unique reports {dropped_no_natural_hazard.shape[0]} dropped because of no natural hazard")
print(f"Number of unique appealCodes {dropped_no_natural_hazard['appealCode'].nunique()} dropped because of no natural hazard")

df_dropped_no_text = df_dropped_no_natural_hazard.loc[df_dropped_no_natural_hazard["processing_error"] != "no text"]

Number of unique reports 471 dropped because of no natural hazard
Number of unique appealCodes 471 dropped because of no natural hazard


In [144]:
print(f"Number of unique reports {df_dropped_no_text.shape[0]} after dropping reports with invalid appealType")
print(f"Number of unique appealCodes {df_dropped_no_text['appealCode'].nunique()} after dropping reports with invalid appealType")

Number of unique reports 2136 after dropping reports with invalid appealType
Number of unique appealCodes 778 after dropping reports with invalid appealType
